# 02 · Chunk + Ingest（写入 Chroma）

目标：
- 先按逻辑单元合并 Markdown，而不是直接对零散页面文件切 chunk
- 复用课件里推荐的 **MarkdownHeaderTextSplitter + RecursiveCharacterTextSplitter**
- 同时构建 **section-level** 和 **chunk-level** 两层索引
- 将结果写入本地 **ChromaDB**（persistent）

输入：上一节解析得到的 Markdown（MinerU 或 PaddleOCR-VL 都可以）


In [1]:
from __future__ import annotations

import os
from pathlib import Path


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")


PROJECT_ROOT = resolve_project_root()

# 选择解析来源：mineru / paddleocr_vl
SOURCE = os.getenv("PARSE_SOURCE", "paddleocr_vl")
BASE_DIR = PROJECT_ROOT / "data/parsed/道通24年年报"

assert BASE_DIR.exists(), f"请先运行上一节解析：{BASE_DIR}"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE:", BASE_DIR)
print("SOURCE:", SOURCE)


PROJECT_ROOT: /Users/mengbai/Documents/AI-training/RAG_project
BASE: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报
SOURCE: paddleocr_vl


### 1) 读取 Markdown，并先整体合并成一个 Markdown

In [2]:


import html
import re
from collections import defaultdict

from langchain_core.documents import Document


def infer_page_span(md_file: Path, source: str) -> tuple[int | None, int | None]:
    if source == "paddleocr_vl":
        group_match = re.search(r"_p(\d+)-(\d+)", md_file.parent.name)
        idx_match = re.search(r"_(\d+)$", md_file.stem)
        if group_match and idx_match:
            start = int(group_match.group(1))
            end = int(group_match.group(2))
            offset = int(idx_match.group(1))
            page_num = start + offset
            if start <= page_num <= end:
                return page_num, page_num
    else:
        page_match = re.search(r"(?:page|p)(\d+)", md_file.stem, flags=re.IGNORECASE)
        if page_match:
            page_num = int(page_match.group(1))
            return page_num, page_num
    return None, None



def rewrite_image_paths(markdown_text: str, relative_path: Path, source: str) -> str:
    source_root = Path(source)
    try:
        parent_dir = relative_path.parent.relative_to(source_root).as_posix()
    except ValueError:
        parent_dir = relative_path.parent.as_posix()

    def replace_src(match):
        original = match.group(1)
        if original.startswith(("http://", "https://", "/", "data:")):
            return match.group(0)
        rewritten = f"{parent_dir}/{original}" if parent_dir and parent_dir != "." else original
        rewritten = rewritten.replace("//", "/")
        return match.group(0).replace(original, rewritten, 1)

    return re.sub(r'src=["\']([^"\']+)["\']', replace_src, markdown_text)



def clean_markdown_for_ingestion(markdown_text: str) -> str:
    """清理 HTML/CSS 噪声，尽量保留正文与表格文本语义。"""
    text = html.unescape(markdown_text)

    # 图片本身对文本 embedding 帮助有限，删除标签避免噪声。
    text = re.sub(r"<img[^>]*>", " ", text, flags=re.IGNORECASE)

    # 去掉常见样式属性，避免 style/class 等污染 chunk 文本。
    text = re.sub(
        r"\s(?:style|class|width|height|align|valign|border|cellpadding|cellspacing)=(\"[^\"]*\"|'[^']*')",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # 将 HTML 表格转成更接近纯文本的结构。
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</?(table|thead|tbody)>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<tr[^>]*>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</tr>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<t[dh][^>]*>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"</t[dh]>", " | ", text, flags=re.IGNORECASE)

    # 常见块级标签换行，剩余标签直接去掉。
    text = re.sub(r"</?(div|p|span|figure|section)[^>]*>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)

    # 收尾清洗：压缩多余分隔符与空白。
    text = re.sub(r"(?:\s*\|\s*){2,}", " | ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()



def load_markdown_documents(source: str) -> list[Document]:
    if source == "mineru":
        md_files = sorted((BASE_DIR / "mineru").glob("*.md"))
        assert md_files, "没找到 mineru 的 md（检查上一节输出 data/parsed/道通24年年报/mineru）"
    elif source == "paddleocr_vl":
        # PaddleOCR-VL 输出是按批次子目录拆开的，这里递归读取全部 md 文档块。
        md_files = sorted((BASE_DIR / "paddleocr_vl").glob("*/*.md"))
        assert md_files, "没找到 paddleocr_vl 的 md（检查上一节输出 data/parsed/道通24年年报/paddleocr_vl/**.md）"
    else:
        raise ValueError(f"Unknown source: {source}")

    documents: list[Document] = []
    for md_file in md_files:
        relative_path = md_file.relative_to(BASE_DIR)
        page_start, page_end = infer_page_span(md_file, source)
        metadata = {
            "source": "道通24年年报",
            "parse_source": source,
            "doc_group": md_file.parent.name,
            "file_name": md_file.name,
            "file_path": relative_path.as_posix(),
            "doc_id": md_file.stem,
            "page_start": page_start,
            "page_end": page_end,
        }
        documents.append(
            Document(
                page_content=rewrite_image_paths(md_file.read_text(encoding="utf-8"), relative_path, source),
                metadata=metadata,
            )
        )

    return documents



def merge_documents_to_single_markdown(documents: list[Document]) -> Document:
    ordered_docs = sorted(
        documents,
        key=lambda d: (
            d.metadata.get("page_start") is None,
            d.metadata.get("page_start") or 10**9,
            d.metadata.get("file_name", ""),
        ),
    )
    merged_text = "\n\n".join(doc.page_content.strip() for doc in ordered_docs if doc.page_content.strip())
    cleaned_text = clean_markdown_for_ingestion(merged_text)
    return Document(
        page_content=cleaned_text,
        metadata={
            "source": ordered_docs[0].metadata["source"],
            "parse_source": ordered_docs[0].metadata["parse_source"],
            "doc_group": "full_document",
            "doc_id": f"{ordered_docs[0].metadata['source']}__full",
            "file_path": f"{ordered_docs[0].metadata['parse_source']}/autel_annual_report_2024.md",
            "page_start": min((doc.metadata.get("page_start") for doc in ordered_docs if doc.metadata.get("page_start") is not None), default=None),
            "page_end": max((doc.metadata.get("page_end") for doc in ordered_docs if doc.metadata.get("page_end") is not None), default=None),
            "source_doc_count": len(ordered_docs),
        },
    )


raw_docs = load_markdown_documents(SOURCE)
merged_doc = merge_documents_to_single_markdown(raw_docs)
merged_docs = [merged_doc]
raw_chars = sum(len(doc.page_content) for doc in raw_docs)
merged_chars = len(merged_doc.page_content)
removed_chars = raw_chars - merged_chars

merged_md_path = BASE_DIR / SOURCE / "autel_annual_report_2024.md"
merged_md_path.write_text(merged_doc.page_content, encoding="utf-8")

print("raw docs:", len(raw_docs))
print("merged docs:", len(merged_docs))
print("raw chars:", raw_chars)
print("merged chars:", merged_chars)
print("removed chars during cleaning:", removed_chars)
print("merged md path:", merged_md_path)
print("sample merged meta:", merged_doc.metadata)
print(merged_doc.page_content[:500])


raw docs: 306
merged docs: 1
raw chars: 877804
merged chars: 878155
merged md path: /Users/mengbai/Documents/AI-training/RAG_project/data/parsed/道通24年年报/paddleocr_vl/autel_annual_report_2024.md
sample merged meta: {'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': 'full_document', 'doc_id': '道通24年年报__full', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'page_start': 1, 'page_end': 306, 'source_doc_count': 306}
# 2024 年度报告

深圳市道通科技股份有限公司

<div style="text-align: center;"><img src="道通24年年报_p0001-0050/imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>

### 拥抱 AI，筑梦未来

#### ——董事长致投资人的一封信

##### 尊敬的投资人朋友：

岁月不居，创新不息。

2025 年，道通科技迎来公司成立 20 周年、上市 5 周年的重要时刻。在此历史节点，我们怀揣感恩与敬畏——感恩股东们的信任与陪伴，敬畏时代赋予我们的使命与责任。回望征程，我们以产品为舟、创新为帆，穿越周期，从行业探索者迈向全球引领者。如今，AI 大潮浩浩荡荡，第四次工业革命奔涌而来，时代的风口正托举着我们驶向星辰大海。

## 2024年：AI深度赋能，业务实现跨越式增长

2024 年，是道通科技全面拥抱 AI 的关键之年。我们以 AI 重构产业价值，以生态协同共创增长，凭借 “人工智能+垂直场景” 战略，实现营收 39.3


### 2) Chunk（先按标题形成 section，再对 section 做递归切细）

In [3]:


from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [("#", "h1"), ("##", "h2"), ("###", "h3")]
header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

child_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", ". ", " ", ""],
    chunk_size=800,
    chunk_overlap=120,
)


def resolve_section_title(metadata: dict) -> str:
    for key in ("h3", "h2", "h1"):
        value = metadata.get(key)
        if value:
            return value
    return metadata.get("doc_group", "untitled")


section_docs = []
chunks = []
for merged_doc in merged_docs:
    header_docs = header_splitter.split_text(merged_doc.page_content)
    if not header_docs:
        header_docs = [Document(page_content=merged_doc.page_content, metadata={})]

    section_docs_for_parent = []
    for section_in_doc, doc in enumerate(header_docs):
        section_metadata = {
            **merged_doc.metadata,
            **doc.metadata,
            "section_in_doc": section_in_doc,
            "section_id": f"{merged_doc.metadata['doc_id']}::section_{section_in_doc}",
            "section_title": resolve_section_title({**merged_doc.metadata, **doc.metadata}),
            "content_level": "section",
        }
        section_doc = Document(page_content=doc.page_content, metadata=section_metadata)
        section_docs.append(section_doc)
        section_docs_for_parent.append(section_doc)

    section_chunk_counts: dict[str, int] = defaultdict(int)
    doc_chunks = child_splitter.split_documents(section_docs_for_parent)
    for chunk_in_doc, doc_chunk in enumerate(doc_chunks):
        section_id = doc_chunk.metadata["section_id"]
        chunk_in_section = section_chunk_counts[section_id]
        section_chunk_counts[section_id] += 1
        doc_chunk.metadata.update(
            {
                "chunk_in_doc": chunk_in_doc,
                "chunk_in_section": chunk_in_section,
                "content_level": "chunk",
            }
        )
        chunks.append(doc_chunk)

for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("section docs:", len(section_docs))
print("chunks:", len(chunks))
print("sample section meta:", section_docs[0].metadata)
print("sample chunk meta:", chunks[0].metadata)
print("sample chunk text:\n", chunks[0].page_content[:300])


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


section docs: 668
chunks: 1979
sample section meta: {'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': 'full_document', 'doc_id': '道通24年年报__full', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'page_start': 1, 'page_end': 306, 'source_doc_count': 306, 'h1': '2024 年度报告', 'section_in_doc': 0, 'section_id': '道通24年年报__full::section_0', 'section_title': '2024 年度报告', 'content_level': 'section'}
sample chunk meta: {'source': '道通24年年报', 'parse_source': 'paddleocr_vl', 'doc_group': 'full_document', 'doc_id': '道通24年年报__full', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'page_start': 1, 'page_end': 306, 'source_doc_count': 306, 'h1': '2024 年度报告', 'section_in_doc': 0, 'section_id': '道通24年年报__full::section_0', 'section_title': '2024 年度报告', 'content_level': 'chunk', 'chunk_in_doc': 0, 'chunk_in_section': 0, 'chunk_id': 0}
sample chunk text:
 深圳市道通科技股份有限公司  
<div style="text-align: center;"><img src="道通24年年报_p0001-0050/imgs/img_in_image_box_4_608_1191_1678.jpg"

### 3) 写入 Chroma（本地持久化）：section-level + chunk-level 两层索引

In [4]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
CHUNK_COLLECTION = "autel_annual_report_2024"
SECTION_COLLECTION = "autel_annual_report_2024_sections"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
embed_model = os.getenv("EMBED_MODEL")

assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

# OpenRouter 兼容 OpenAI SDK，这里显式传入 api_key/base_url。
emb = OpenAIEmbeddings(
    model=embed_model,
    api_key=openai_api_key,
    base_url=openai_base_url,
)


import chromadb

_chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))


def rebuild_collection(collection_name: str, docs: list[Document], ids: list[str]):
    try:
        _chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    vectordb = Chroma(
        collection_name=collection_name,
        embedding_function=emb,
        client=_chroma_client,
    )
    vectordb.add_documents(docs, ids=ids)
    return vectordb


section_ids = [doc.metadata["section_id"] for doc in section_docs]
chunk_ids = [
    f"{doc.metadata['section_id']}::chunk_{doc.metadata['chunk_in_section']}"
    for doc in chunks
]

section_vs = rebuild_collection(SECTION_COLLECTION, section_docs, section_ids)
chunk_vs = rebuild_collection(CHUNK_COLLECTION, chunks, chunk_ids)

print("persisted:", CHROMA_DIR.resolve())
print("chunk collection:", CHUNK_COLLECTION)
print("section collection:", SECTION_COLLECTION)
print("embed:", embed_model)
print("env:", ENV_FILE)
print("raw docs ingested:", len(raw_docs))
print("merged docs ingested:", len(merged_docs))
print("section docs ingested:", len(section_docs))
print("chunks ingested:", len(chunks))
print("sample rebuilt chunk id:", chunk_ids[0])


/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_55059/462965186.py:44: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


persisted: /Users/mengbai/Documents/AI-training/RAG_project/data/chroma
chunk collection: autel_annual_report_2024
section collection: autel_annual_report_2024_sections
embed: Qwen/Qwen3-Embedding-8B
env: /Users/mengbai/Documents/AI-training/.env
raw docs ingested: 306
merged docs ingested: 1
section docs ingested: 668
chunks ingested: 1979
sample rebuilt chunk id: 道通24年年报__full::section_0::chunk_0


## 查看两层索引实际存储结果

这里直接分别连接 `section-level` 和 `chunk-level` 两个 Chroma collection，看它们各自保存的 `id`、`metadata` 和正文片段。这样你可以直接确认：
- 粗粒度层是否保留了完整章节语义
- 细粒度层是否带着 `section_id` 等父级信息
- 两层索引的命名和字段是否符合预期

In [ ]:
from pprint import pprint

section_vs = Chroma(
    collection_name=SECTION_COLLECTION,
    embedding_function=emb,
    client=_chroma_client,
)
chunk_vs = Chroma(
    collection_name=CHUNK_COLLECTION,
    embedding_function=emb,
    client=_chroma_client,
)


def preview_collection(vs, label: str, limit: int = 3, text_len: int = 400):
    raw = vs.get(limit=limit)
    docs = raw.get("documents", [])
    metas = raw.get("metadatas", [])
    ids = raw.get("ids", [])

    print(f"\n===== {label} =====")
    print("items:", len(docs))
    for i, (doc_id, doc, meta) in enumerate(zip(ids, docs, metas), 1):
        print(f"\n--- item {i} ---")
        print("id:", doc_id)
        print("metadata:")
        pprint(meta)
        print("text:")
        print(doc[:text_len].replace("\n", " "))


print(preview_collection(section_vs, "section-level collection", limit=3))
print(preview_collection(chunk_vs, "chunk-level collection", limit=3))



===== section-level collection =====
items: 3

--- item 1 ---
id: 道通24年年报__full::section_0
metadata:
{'content_level': 'section',
 'doc_group': 'full_document',
 'doc_id': '道通24年年报__full',
 'file_path': 'paddleocr_vl/autel_annual_report_2024.md',
 'h1': '2024 年度报告',
 'page_end': 306,
 'page_start': 1,
 'parse_source': 'paddleocr_vl',
 'section_id': '道通24年年报__full::section_0',
 'section_in_doc': 0,
 'section_title': '2024 年度报告',
 'source': '道通24年年报',
 'source_doc_count': 306}
text:
深圳市道通科技股份有限公司   <div style="text-align: center;"><img src="道通24年年报_p0001-0050/imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>

--- item 2 ---
id: 道通24年年报__full::section_1
metadata:
{'content_level': 'section',
 'doc_group': 'full_document',
 'doc_id': '道通24年年报__full',
 'file_path': 'paddleocr_vl/autel_annual_report_2024.md',
 'h1': '2024 年度报告',
 'h3': '拥抱 AI，筑梦未来',
 'page_end': 306,
 'page_start': 1,
 'parse_source': 'paddleocr_vl',
 'section_id': '道通24年年报__full::section_1',
 'sec

## 导出 section -> chunks 父子视图

这个导出会把 `section-level` 和 `chunk-level` 按 `section_id` 对齐，生成一个适合人工检查的 Markdown 文件。你可以直接看每个 section 下面挂了哪些 chunk，确认层级关系是不是对的。

In [13]:
from collections import defaultdict
import json

def export_parent_child_view(
    section_vs,
    chunk_vs,
    output_path: Path,
    max_sections: int | None = 30,
    max_chunks_per_section: int = 8,
    text_len: int = 500,
):
    section_raw = section_vs.get()
    chunk_raw = chunk_vs.get()

    section_rows = [
        {"id": doc_id, "metadata": meta, "text": doc}
        for doc_id, doc, meta in zip(
            section_raw.get("ids", []),
            section_raw.get("documents", []),
            section_raw.get("metadatas", []),
        )
    ]
    chunk_rows = [
        {"id": doc_id, "metadata": meta, "text": doc}
        for doc_id, doc, meta in zip(
            chunk_raw.get("ids", []),
            chunk_raw.get("documents", []),
            chunk_raw.get("metadatas", []),
        )
    ]

    chunks_by_section: dict[str, list[dict]] = defaultdict(list)
    for row in chunk_rows:
        section_id = row["metadata"].get("section_id", "__missing_section_id__")
        chunks_by_section[section_id].append(row)

    for rows in chunks_by_section.values():
        rows.sort(key=lambda r: (r["metadata"].get("chunk_in_section", 10**9), r["id"]))

    section_rows.sort(key=lambda r: (r["metadata"].get("section_in_doc", 10**9), r["id"]))
    if max_sections is not None:
        section_rows = section_rows[:max_sections]

    with output_path.open("w", encoding="utf-8") as f:
        f.write("# section -> chunks parent-child view\n\n")
        f.write(f"total_sections_exported: {len(section_rows)}\n\n")

        for i, section in enumerate(section_rows, 1):
            section_id = section["metadata"].get("section_id", section["id"])
            children = chunks_by_section.get(section_id, [])

            f.write(f"## section {i}: {section_id}\n\n")
            f.write(f"- section_title: `{section['metadata'].get('section_title', '')}`\n")
            f.write(f"- child_chunk_count: `{len(children)}`\n")
            f.write(f"- metadata: `{json.dumps(section['metadata'], ensure_ascii=False)}`\n\n")
            f.write("### section text\n\n")
            f.write("```text\n")
            f.write(section["text"][:text_len].replace("```", "'''"))
            f.write("\n```\n\n")

            f.write("### child chunks\n\n")
            for j, child in enumerate(children[:max_chunks_per_section], 1):
                f.write(f"#### chunk {j}: `{child['id']}`\n\n")
                f.write(f"- chunk_in_section: `{child['metadata'].get('chunk_in_section')}`\n")
                f.write(f"- metadata: `{json.dumps(child['metadata'], ensure_ascii=False)}`\n\n")
                f.write("```text\n")
                f.write(child["text"][:text_len].replace("```", "'''"))
                f.write("\n```\n\n")

            if len(children) > max_chunks_per_section:
                f.write(f"... truncated {len(children) - max_chunks_per_section} more chunks ...\n\n")

    print("parent-child view:", output_path)
    print("sections exported:", len(section_rows))


EXPORT_DIR = globals().get("EXPORT_DIR", PROJECT_ROOT / "data/chroma_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

parent_child_path = EXPORT_DIR / "autel_annual_report_2024_parent_child_view.md"
export_parent_child_view(section_vs, chunk_vs, parent_child_path)


parent-child view: /Users/mengbai/Documents/AI-training/RAG_project/data/chroma_exports/autel_annual_report_2024_parent_child_view.md
sections exported: 30



===== chunk-level collection =====
items: 10

--- item 1 ---
id: 道通24年年报__full::section_0::chunk_0
metadata:
{'chunk_id': 0,
 'chunk_in_doc': 0,
 'chunk_in_section': 0,
 'content_level': 'chunk',
 'doc_group': 'full_document',
 'doc_id': '道通24年年报__full',
 'file_path': 'paddleocr_vl/autel_annual_report_2024.md',
 'h1': '2024 年度报告',
 'page_end': 306,
 'page_start': 1,
 'parse_source': 'paddleocr_vl',
 'section_id': '道通24年年报__full::section_0',
 'section_in_doc': 0,
 'section_title': '2024 年度报告',
 'source': '道通24年年报',
 'source_doc_count': 306}
text:
深圳市道通科技股份有限公司   <div style="text-align: center;"><img src="道通24年年报_p0001-0050/imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>

--- item 2 ---
id: 道通24年年报__full::section_1::chunk_0
metadata:
{'chunk_id': 1,
 'chunk_in_doc': 1,
 'chunk_in_section': 0,
 'content_level': 'chunk',
 'doc_group': 'full_document',
 'doc_id': '道通24年年报__full',
 'file_path': 'paddleocr_vl/autel_annual_report_2024.md',
 'h1': '2024 年度报告',
 'h3'